In [0]:
from pyspark.sql import functions as F

CATALOG = "vcredit_medallion"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

In [0]:
dim_motivo = spark.table(f"{SILVER_SCHEMA}.tb_base_motivos").select(
    "id_motivo",
    F.col("nome_motivo").alias("motivo"),
    "categoria",
    "criticidade",
)

dim_canal = spark.table(f"{SILVER_SCHEMA}.tb_canais").select(
    "id_canal",
    F.col("nome_canal").alias("canal"),
    "tipo_canal",
)

for df, nome in [(dim_motivo, "dim_motivo"), (dim_canal, "dim_canal")]:
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.{nome}")
    )
    print(nome, spark.table(f"{GOLD_SCHEMA}.{nome}").count())

dim_motivo 14
dim_canal 4


In [0]:
dim_atendente = spark.table(f"{SILVER_SCHEMA}.tb_base_atendentes").select(
    "id_atendente",
    "nome_atendente",
    "nivel_atendimento",
    "data_admissao",
    "data_desligamento",
    "status_atendente",
)

dim_cliente = spark.table(f"{SILVER_SCHEMA}.tb_clientes").select(
    "id_cliente",
    "regiao",
    "idade",
    "data_cadastro",
    "dia_vencimento_fatura",
)

for df, nome in [(dim_atendente, "dim_atendente"), (dim_cliente, "dim_cliente")]:
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.{nome}")
    )
    print(nome, spark.table(f"{GOLD_SCHEMA}.{nome}").count())

dim_atendente 25
dim_cliente 47705


In [0]:
from itertools import chain

DATA_INICIO = "2025-01-01"
DATA_FIM = "2026-07-31"

nomes_mes = {1: "janeiro", 2: "fevereiro", 3: "março", 4: "abril", 5: "maio", 6: "junho",
             7: "julho", 8: "agosto", 9: "setembro", 10: "outubro", 11: "novembro", 12: "dezembro"}
nomes_dia = {1: "domingo", 2: "segunda", 3: "terça", 4: "quarta", 5: "quinta", 6: "sexta", 7: "sábado"}

# Transforma os dicionários em mapas do Spark para consultar pelo número
mapa_mes = F.create_map([F.lit(x) for x in chain(*nomes_mes.items())])
mapa_dia = F.create_map([F.lit(x) for x in chain(*nomes_dia.items())])

dim_data = (
    spark.sql(f"""
        SELECT explode(sequence(to_date('{DATA_INICIO}'), to_date('{DATA_FIM}'), interval 1 day)) AS data
    """)
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("nome_mes", mapa_mes[F.month("data")])
    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("dia_semana", F.dayofweek("data"))  # 1 = domingo, 7 = sábado
    .withColumn("nome_dia_semana", mapa_dia[F.dayofweek("data")])
    .withColumn("fim_de_semana", F.dayofweek("data").isin(1, 7))
)

(
    dim_data.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.dim_data")
)
print("dim_data", spark.table(f"{GOLD_SCHEMA}.dim_data").count())

dim_data 577


In [0]:
custos_por_chamado = (
    spark.table(f"{SILVER_SCHEMA}.tb_custos")
    .withColumn("id_chamado", F.expr("try_cast(id_chamado as bigint)"))
    .groupBy("id_chamado")
    .agg(
        F.sum("custo").alias("custo_total"),
        F.count("*").alias("qtd_lancamentos_custo"),
        F.sum(F.col("custo_negativo").cast("int")).alias("qtd_custos_negativos"),
    )
)

pesquisas_por_chamado = (
    spark.table(f"{SILVER_SCHEMA}.tb_pesquisa_satisfacao")
    .withColumn("id_chamado", F.expr("try_cast(id_chamado as bigint)"))
    .groupBy("id_chamado")
    .agg(
        # when sem otherwise devolve nulo pras notas inválidas, e o avg ignora nulos
        F.avg(F.when(F.col("nota_valida"), F.col("nota_atendimento"))).alias("nota_media"),
        F.count("*").alias("qtd_pesquisas"),
    )
)

# Conferências: a soma não pode mudar e cada chamado tem que aparecer uma vez só
soma_silver = spark.table(f"{SILVER_SCHEMA}.tb_custos").agg(F.sum("custo")).first()[0]
soma_agregada = custos_por_chamado.agg(F.sum("custo_total")).first()[0]
print("soma custos silver:", soma_silver, "| agregada:", soma_agregada)

for nome, df in [("custos", custos_por_chamado), ("pesquisas", pesquisas_por_chamado)]:
    linhas = df.count()
    ids = df.select("id_chamado").distinct().count()
    nulos = df.filter(F.col("id_chamado").isNull()).count()
    print(f"{nome}: {linhas} chamados | ids distintos: {ids} | id nulo: {nulos}")

soma custos silver: 1447671.52 | agregada: 1447671.52
custos: 130288 chamados | ids distintos: 130288 | id nulo: 0
pesquisas: 80219 chamados | ids distintos: 80219 | id nulo: 0


In [0]:
chamados = spark.table(f"{SILVER_SCHEMA}.tb_chamados").select(
    "id_chamado",
    F.col("id_cliente").alias("id_cliente_chamados"),
    "id_atendente",
    "id_motivo",
    "id_canal",
    "resolvido",
)

hora = spark.table(f"{SILVER_SCHEMA}.tb_chamados_hora").select(
    F.expr("try_cast(id_chamado as bigint)").alias("id_chamado"),
    # Texto vazio ou "null" vira nulo de verdade
    F.when(~F.lower(F.trim("id_cliente")).isin("", "null"), F.trim("id_cliente")).alias("id_cliente_hora"),
    "hora_abertura_chamado",
    "abertura_no_periodo",
    "ordem_temporal_invalida",
    "tempo_espera_segundos",
    "tempo_atendimento_segundos",
    "tempo_total_segundos",
)

# Todos os joins são left a partir de chamados e todas as chaves da direita são únicas,
# então nenhum chamado some nem se repete aqui
base = (
    chamados
    .join(hora, "id_chamado", "left")
    .join(custos_por_chamado, "id_chamado", "left")
    .join(pesquisas_por_chamado, "id_chamado", "left")
)

fato_chamados = base.filter(F.col("abertura_no_periodo") == True).select(
    "id_chamado",
    F.coalesce("id_cliente_hora", "id_cliente_chamados").alias("id_cliente"),
    "id_atendente",
    "id_motivo",
    "id_canal",
    F.to_date("hora_abertura_chamado").alias("data_abertura"),
    F.hour("hora_abertura_chamado").alias("hora_do_dia"),
    "hora_abertura_chamado",
    "resolvido",
    "tempo_espera_segundos",
    "tempo_atendimento_segundos",
    "tempo_total_segundos",
    "ordem_temporal_invalida",
    # Sem lançamento de custo fica nulo, que é diferente de custo zero
    "custo_total",
    F.coalesce("qtd_lancamentos_custo", F.lit(0)).alias("qtd_lancamentos_custo"),
    F.coalesce("qtd_custos_negativos", F.lit(0)).alias("qtd_custos_negativos"),
    "nota_media",
    F.coalesce("qtd_pesquisas", F.lit(0)).alias("qtd_pesquisas"),
)

# Conferências antes de gravar
total_chamados = chamados.count()
total_base = base.count()
sem_horario = base.filter(F.col("hora_abertura_chamado").isNull()).count()
fora_periodo = base.filter(F.col("hora_abertura_chamado").isNotNull() & ~F.col("abertura_no_periodo")).count()
total_fato = fato_chamados.count()
ids_fato = fato_chamados.select("id_chamado").distinct().count()

print(f"chamados silver: {total_chamados} | após joins: {total_base}")
print(f"sem horário: {sem_horario} | fora do período: {fora_periodo} | fato: {total_fato}")
print(f"ids distintos na fato: {ids_fato}")

assert total_base == total_chamados, "Algum join duplicou ou perdeu chamados"
assert total_fato == ids_fato, "id_chamado repetido na fato"
assert total_fato > 0, "Fato vazia"

(
    fato_chamados.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_SCHEMA}.fato_chamados")
)
print("fato_chamados gravada:", spark.table(f"{GOLD_SCHEMA}.fato_chamados").count())

chamados silver: 131838 | após joins: 131838
sem horário: 0 | fora do período: 198 | fato: 131640
ids distintos na fato: 131640
fato_chamados gravada: 131640


In [0]:
fato = spark.table(f"{GOLD_SCHEMA}.fato_chamados")

checagens = [
    ("id_cliente", "dim_cliente"),
    ("id_atendente", "dim_atendente"),
    ("id_motivo", "dim_motivo"),
    ("id_canal", "dim_canal"),
]

for chave, dim in checagens:
    dim_df = spark.table(f"{GOLD_SCHEMA}.{dim}").select(chave)
    nulos = fato.filter(F.col(chave).isNull()).count()
    sem_par = fato.filter(F.col(chave).isNotNull()).join(dim_df, chave, "left_anti").count()
    print(f"{chave}: nulo {nulos} | sem par na {dim}: {sem_par}")

dim_data = spark.table(f"{GOLD_SCHEMA}.dim_data").select(F.col("data").alias("data_abertura"))
print("data_abertura sem par na dim_data:", fato.join(dim_data, "data_abertura", "left_anti").count())


id_cliente: nulo 0 | sem par na dim_cliente: 0
id_atendente: nulo 761 | sem par na dim_atendente: 0
id_motivo: nulo 0 | sem par na dim_motivo: 0
id_canal: nulo 0 | sem par na dim_canal: 0
data_abertura sem par na dim_data: 0
